# Pramana experiment runner and reusable report

Edit the configuration below to launch either a deterministic direct run or an intent-driven run. Each launch creates a new result folder and its own `report.html`. Set `RUN_EXPERIMENT = False` and `RUN_DIR` to re-render an existing folder.

In [ ]:
# This cell is the normal experiment control surface.
RUN_EXPERIMENT = False
MODE = 'deterministic'  # 'deterministic' or 'intent'
APPS = ['youtube', 'vimeo']  # also supports ['youtube', 'tubi']
CAPACITIES_MBPS = [3, 6, 10]
DURATION_SECONDS = 30
LATENCY_MS = 100
BROWSER_RESOLUTION = '1080p'
INTENT = 'Run YouTube and Vimeo concurrently at 6 Mbps and 100 ms for 30 seconds'
RUN_DIR = ''  # Used only when RUN_EXPERIMENT is False.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
pramana_dir = repo_root / 'experiments' / 'pramana'

# Runner-created HTML supplies this variable. It prevents recursive launches.
managed_run_dir = os.environ.get('PRAMANA_RUN_DIR', '')
if managed_run_dir:
    run_dir = Path(managed_run_dir).expanduser().resolve()
elif RUN_EXPERIMENT:
    if MODE == 'deterministic':
        command = [sys.executable, str(pramana_dir / 'run_pramana_direct.py'),
                   '--apps', *APPS, '--capacities-mbps', *map(str, CAPACITIES_MBPS),
                   '--duration-seconds', str(DURATION_SECONDS),
                   '--latency-ms', str(LATENCY_MS),
                   '--browser-resolution', BROWSER_RESOLUTION]
        marker = 'Pramana run: '
    elif MODE == 'intent':
        command = [sys.executable, str(pramana_dir / 'run_pramana_intent.py'), INTENT]
        marker = 'Pramana intent run: '
    else:
        raise ValueError("MODE must be 'deterministic' or 'intent'")
    completed = subprocess.run(command, cwd=repo_root, text=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, check=True)
    print(completed.stdout)
    paths = [line.removeprefix(marker) for line in completed.stdout.splitlines() if line.startswith(marker)]
    if not paths:
        raise RuntimeError('Runner completed without reporting its result directory')
    run_dir = Path(paths[-1]).resolve()
else:
    if not RUN_DIR:
        raise ValueError('Set RUN_EXPERIMENT=True, or paste an existing folder into RUN_DIR')
    run_dir = Path(RUN_DIR).expanduser().resolve()

run_dir


In [ ]:
sys.path.insert(0, str(pramana_dir))
from pramana_report import render_report

title = os.environ.get('PRAMANA_REPORT_TITLE', f'Pramana report — {run_dir.name}')
report_context = render_report(run_dir, title)
